In [3]:
import cv2
import os
import numpy as np
import pyodbc
from datetime import datetime
import time
import pandas as pd
import subprocess

# Conexión a SQL Server
conn = pyodbc.connect('DRIVER={SQL Server};SERVER=ASUS1\\SQLEXPRESS;DATABASE=EmpresaDeteccionFacial;Trusted_Connection=yes')
#conn = pyodbc.connect('DRIVER={SQL Server};SERVER=DESKTOP-1P4IV38;DATABASE=EmpresaDeteccionFacial;Trusted_Connection=yes')
cursor = conn.cursor()

# Clasificador Haar
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

# Leer rostros conocidos y etiquetas
base_path = 'rostros_conocidos'
rostros_conocidos = []
etiquetas = []
nombre_por_etiqueta = {}
etiqueta_por_nombre = {}
etiqueta_id = 0

for persona in os.listdir(base_path):
    persona_path = os.path.join(base_path, persona)
    if os.path.isdir(persona_path):
        for archivo in os.listdir(persona_path):
            img_path = os.path.join(persona_path, archivo)
            imagen = cv2.imread(img_path)
            if imagen is None:
                continue
            gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
            gris = cv2.equalizeHist(gris)
            rostros = face_cascade.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=4, minSize=(80, 80))
            if len(rostros) == 0:
                print(f"⚠️ No se detectó rostro en: {img_path}")
                continue
            for (x, y, w, h) in rostros:
                rostro = gris[y:y+h, x:x+w]
                rostro = cv2.resize(rostro, (150, 150))
                if persona not in etiqueta_por_nombre:
                    etiqueta_por_nombre[persona] = etiqueta_id
                    nombre_por_etiqueta[etiqueta_id] = persona
                    etiqueta_id += 1
                rostros_conocidos.append(rostro)
                etiquetas.append(etiqueta_por_nombre[persona])
                break

# Entrenar modelo LBPH
face_recognizer = cv2.face.LBPHFaceRecognizer_create()
face_recognizer.train(rostros_conocidos, np.array(etiquetas))

# Funciones auxiliares

def exportar_a_excel(nombre):
    cursor.execute("""
        SELECT TOP 5 IdRegistro, NombresP, ApellidosP, FechaHoraIngreso, FechaHoraSalida
        FROM Personal
        WHERE NombresP = ? AND ApellidosP = ?
        ORDER BY FechaHoraIngreso DESC
    """, nombre.split()[0], ' '.join(nombre.split()[1:]))
    registros = cursor.fetchall()
    datos = []
    for reg in registros:
        datos.append({
            "ID": reg.IdRegistro,
            "Nombre": reg.NombresP,
            "Apellido": reg.ApellidosP,
            "Ingreso": reg.FechaHoraIngreso.strftime("%Y-%m-%d %H:%M:%S") if reg.FechaHoraIngreso else "",
            "Salida": reg.FechaHoraSalida.strftime("%Y-%m-%d %H:%M:%S") if reg.FechaHoraSalida else ""
        })
    df = pd.DataFrame(datos)
    archivo = f"reporte_{nombre.replace(' ', '_')}.xlsx"
    df.to_excel(archivo, index=False)
    print(f"✅ Exportado: {archivo}")
    try:
        subprocess.Popen(["start", "excel", archivo], shell=True)
    except:
        pass

def obtener_registros(nombre):
    cursor.execute("""
        SELECT TOP 5 IdRegistro, FechaHoraIngreso, FechaHoraSalida
        FROM Personal
        WHERE NombresP = ? AND ApellidosP = ?
        ORDER BY FechaHoraIngreso DESC
    """, nombre.split()[0], ' '.join(nombre.split()[1:]))
    registros = cursor.fetchall()
    textos = []
    for reg in registros:
        texto = f"ID: {reg.IdRegistro} | In: {reg.FechaHoraIngreso.strftime('%H:%M:%S') if reg.FechaHoraIngreso else ''} | Out: {reg.FechaHoraSalida.strftime('%H:%M:%S') if reg.FechaHoraSalida else ''}"
        textos.append(texto)
    return textos

def obtener_estado_actual(nombre):
    cursor.execute("""
        SELECT TOP 1 FechaHoraIngreso, FechaHoraSalida
        FROM Personal
        WHERE NombresP = ? AND ApellidosP = ?
        ORDER BY FechaHoraIngreso DESC
    """, nombre.split()[0], ' '.join(nombre.split()[1:]))
    registro = cursor.fetchone()
    if registro:
        ingreso, salida = registro.FechaHoraIngreso, registro.FechaHoraSalida
        if salida is None:
            return "esperando_salida"
    return "esperando_entrada"

def registrar_ingreso(nombre):
    now = datetime.now()
    cursor.execute("""
        INSERT INTO Personal (NombresP, ApellidosP, FechaHoraIngreso)
        VALUES (?, ?, ?)
    """, nombre.split()[0], ' '.join(nombre.split()[1:]), now)
    conn.commit()
    print(f"🟢 Ingreso registrado para {nombre} a las {now.strftime('%H:%M:%S')}")

def registrar_salida(nombre):
    now = datetime.now()
    cursor.execute("""
        UPDATE Personal
        SET FechaHoraSalida = ?
        WHERE IdRegistro = (
            SELECT TOP 1 IdRegistro
            FROM Personal
            WHERE NombresP = ? AND ApellidosP = ? AND FechaHoraSalida IS NULL
            ORDER BY FechaHoraIngreso DESC
        )
    """, now, nombre.split()[0], ' '.join(nombre.split()[1:]))
    conn.commit()
    print(f"🔴 Salida registrada para {nombre} a las {now.strftime('%H:%M:%S')}")

# Variables de control
reconocido = None
tiempo_inicio = 0
nombre_actual = ""
boton_coords = (10, 10, 200, 50)
boton_presionado = False
escala = 1.5
registro_tiempos = {}
tiempo_espera_segundos = 10

def click_event(event, x, y, flags, param):
    global boton_presionado
    x1, y1, x2, y2 = boton_coords
    if event == cv2.EVENT_LBUTTONDOWN:
        if x1 <= x <= x2 and y1 <= y <= y2:
            boton_presionado = True

cap = cv2.VideoCapture(0)
cv2.namedWindow("Reconocimiento con Botón")
cv2.setMouseCallback("Reconocimiento con Botón", click_event)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gris = cv2.equalizeHist(gris)
    rostros = face_cascade.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    for (x, y, w, h) in rostros:
        rostro_actual = gris[y:y+h, x:x+w]
        rostro_actual = cv2.resize(rostro_actual, (150, 150))

        nombre = "Desconocido"
        id_predicho, confianza = face_recognizer.predict(rostro_actual)

        if confianza < 70:
            nombre = nombre_por_etiqueta[id_predicho]
            print(f"✅ Reconocido: {nombre} (confianza: {confianza:.2f})")
        else:
            print(f"❌ No reconocido (confianza: {confianza:.2f})")

        if nombre != "Desconocido":
            if reconocido == nombre:
                if time.time() - tiempo_inicio >= 3:
                    nombre_actual = nombre
            else:
                reconocido = nombre
                tiempo_inicio = time.time()

            ahora = time.time()
            if nombre not in registro_tiempos or (ahora - registro_tiempos[nombre]) > tiempo_espera_segundos:
                estado = obtener_estado_actual(nombre)
                if estado == "esperando_entrada":
                    registrar_ingreso(nombre)
                    registro_tiempos[nombre] = ahora
                elif estado == "esperando_salida":
                    registrar_salida(nombre)
                    registro_tiempos[nombre] = ahora

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, nombre, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    # Dibujar botón
    x1, y1, x2, y2 = boton_coords
    color_boton = (0, 128, 255) if not boton_presionado else (0, 255, 0)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color_boton, -1)
    cv2.putText(frame, "EXPORTAR EXCEL", (x1 + 10, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    if boton_presionado:
        if nombre_actual != "":
            exportar_a_excel(nombre_actual)
        boton_presionado = False

    # Crear nuevo frame extendido
    extra_width = 600
    altura, ancho = frame.shape[:2]
    nuevo_frame = np.zeros((altura, ancho + extra_width, 3), dtype=np.uint8)
    nuevo_frame[:, :ancho] = frame

    if nombre_actual != "":
        registros_texto = obtener_registros(nombre_actual)
        alto_texto = 20
        x_texto = ancho + 20
        y_texto = 60

        cv2.rectangle(nuevo_frame, (ancho + 10, 10), (ancho + extra_width - 10, 10 + len(registros_texto) * alto_texto + 50), (50, 50, 50), -1)
        cv2.putText(nuevo_frame, f"Registros de {nombre_actual}:", (x_texto, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        for i, texto in enumerate(registros_texto):
            cv2.putText(nuevo_frame, texto, (x_texto, y_texto + i * alto_texto), cv2.FONT_HERSHEY_PLAIN, 1.3, (200, 255, 200), 1)

    nuevo_frame = cv2.resize(nuevo_frame, (0, 0), fx=escala, fy=escala)
    cv2.imshow("Reconocimiento con Botón", nuevo_frame)
    if cv2.waitKey(1) & 0xFF in [ord('q'), ord('Q')]:
        break

cap.release()
cv2.destroyAllWindows()
cursor.close()
conn.close()

AttributeError: module 'cv2' has no attribute 'CascadeClassifier'